# 02 — Huấn luyện MobileNetV3 embedding/classification

Notebook này trình bày hợp đồng dữ liệu và gọi workflow huấn luyện AXE 5 SKU hiện có. Model tạo embedding L2-normalized và head phân loại; cấu hình quyết định chạy `classification_only` hay retrieval. Notebook không chứa vòng lặp train, loss hoặc metric riêng: các phần đó tiếp tục nằm trong `src.training.classification_trainer` và `src.training.retrieval_trainer`.

Gallery hiện chỉ phủ 5 SKU AXE. Không suy diễn rằng 8 thương hiệu detector còn lại đã có item-level coverage. Cosine similarity là độ tương đồng, không phải xác suất đã hiệu chỉnh.

## Khởi chạy

```powershell
Set-Location D:/code/anhoa/train_embedding
./.venv/Scripts/Activate.ps1
jupyter lab notebooks/02_embedding_training.ipynb
```

Chạy `00_dataset_audit.ipynb` trước. Không có ô cài package, tải weights hoặc tự động resume.

In [ ]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import csv
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir() or not (PROJECT_ROOT / "configs").is_dir():
    raise RuntimeError("Hãy khởi chạy Jupyter từ thư mục gốc của repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import Markdown, display

from scripts.train_axe_5sku import safe_run_name, validate_manual_config
from src.catalog.profiles import load_product_profile
from src.config import resolve_repo_path
from src.training.embedding_data import audit_training_readiness, path_key, read_csv
from src.training.embedding_workflow import build_training_plan
from src.training.classification_trainer import ClassificationTrainer
from src.training.retrieval_trainer import RetrievalTrainer
from src.utils.config import load_config

def show_table(rows: list[dict], title: str) -> None:
    display(Markdown(f"**{title}**"))
    if not rows:
        display(Markdown("_Không có dòng dữ liệu._"))
        return
    columns = list(rows[0])
    clean = lambda value: str(value).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(columns) + " |", "|" + "|".join(["---"] * len(columns)) + "|"]
    lines.extend("| " + " | ".join(clean(row.get(column, "")) for column in columns) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

## Chọn thí nghiệm và cờ thực thi

Cấu hình mặc định là thí nghiệm classification 100 epoch đã được hỗ trợ. Có thể chọn `configs/axe_5sku.yaml` để dùng workflow retrieval 15 epoch. Mọi cờ thực thi đều tắt; `val` là split mặc định và test có cờ opt-in riêng.

In [ ]:
TRAINING_CONFIG_PATH = "configs/axe_5sku_100ep_classification.yaml"
PRODUCT_PROFILE_NAME = "axe5"
TRAINING_RUN_NAME = "notebook_axe5_classification_s42"
DEVICE_OVERRIDE = None  # Ví dụ: "cuda" hoặc "cpu"; None dùng YAML.
VALIDATION_RUN_DIR = "outputs/axe_5sku/classification_100ep_s42"
TEST_CHECKPOINT = "checkpoints/axe_5sku/classification_100ep_s42/best_model.pth"
TEST_OUTPUT_DIR = "outputs/notebook_axe5_test_classification"
DEFAULT_EVALUATION_SPLIT = "val"

RUN_READINESS_CHECK = False
RUN_TRAINING = False
RUN_VALIDATION_REVIEW = False
RUN_TEST_EVALUATION = False

assert DEFAULT_EVALUATION_SPLIT == "val"
safe_run_name(TRAINING_RUN_NAME)
show_table(
    [
        {"tham số": "training config", "giá trị": TRAINING_CONFIG_PATH},
        {"tham số": "run name mới", "giá trị": TRAINING_RUN_NAME},
        {"tham số": "split mặc định", "giá trị": DEFAULT_EVALUATION_SPLIT},
        {"tham số": "checkpoint test", "giá trị": TEST_CHECKPOINT},
    ],
    "Lựa chọn hiện tại",
)

## Cấu hình, trainer và checkpoint policy

`validate_manual_config` là guard được CLI dùng trước khi tạo output. Classification chọn checkpoint theo validation Macro F1 rồi CE loss; retrieval chọn validation macro Recall@1. Hai định nghĩa metric không bị trộn lẫn.

In [ ]:
generic_plan = build_training_plan(TRAINING_CONFIG_PATH)
assert generic_plan["evaluation_split"] == "val"
assert generic_plan["test_manifest_loaded"] is False
assert generic_plan["training_started"] is False
config = load_config(TRAINING_CONFIG_PATH)
validate_manual_config(config)
training_mode = str(config["training"].get("mode", "retrieval"))
trainer_class = ClassificationTrainer if training_mode == "classification_only" else RetrievalTrainer

config_rows = [
    {"nhóm": "workflow", "tham số": "mode", "giá trị": training_mode},
    {"nhóm": "workflow", "tham số": "trainer", "giá trị": trainer_class.__name__},
]
for section in ("dataset", "model", "training", "augmentation", "metric_learning", "output"):
    config_rows.extend({"nhóm": section, "tham số": key, "giá trị": value} for key, value in config.get(section, {}).items())
show_table(config_rows, "Cấu hình embedding/classification")

## Ánh xạ checkpoint → item code

Product profile là hợp đồng định danh dùng chung, không suy ra item code từ tên ZIP. Giá trị item code phải là chuỗi để không mất số 0 đầu. Guard Blue Lavender dưới đây ngăn việc nhập nhầm spray và body wash.

In [ ]:
profile = load_product_profile(PRODUCT_PROFILE_NAME)
identity_rows = [
    {
        "class_index": row.class_index,
        "legacy_class_name": row.legacy_class_name,
        "item_code": row.item_code,
        "item_code_type": type(row.item_code).__name__,
    }
    for row in profile.classes
]
show_table(identity_rows, "Checkpoint class order và item code")
guard = profile.identity_guard("blue_lavender")
show_table([{"identity": name, "item_code": code} for name, code in guard.items()], "Blue Lavender identity guard")
assert guard["spray"] == "64823658"
assert guard["body_wash"] == "64863894"
assert guard["spray"] != guard["body_wash"]
assert all(isinstance(row.item_code, str) for row in profile.classes)

## Số lượng và provenance train/validation

Ô này dùng `read_csv`/`path_key` từ module dữ liệu huấn luyện. Nó không mở `test.csv`. Số family và source ZIP giúp nhận ra trường hợp nhiều crop nhưng ít nguồn độc lập.

In [ ]:
crops_root = resolve_repo_path(config["dataset"]["path"])
clean_root = resolve_repo_path(config["dataset"]["source_coco_path"])
manifest_root = resolve_repo_path(config["dataset"]["split_dir"])
manifest_rows = {
    "train": read_csv(manifest_root / "train.csv"),
    "val": read_csv(manifest_root / "val.csv"),
}
crop_rows = read_csv(crops_root / "crop_manifest.csv")
crop_by_path = {path_key(row["generated_crop_path"]): row for row in crop_rows}
count_rows = []
for split, rows in manifest_rows.items():
    counts = Counter(row["class_name"] for row in rows)
    for identity in profile.classes:
        provenance = [crop_by_path[path_key(row["image_path"])] for row in rows if row["class_name"] == identity.legacy_class_name]
        count_rows.append(
            {
                "split": split,
                "class_index": identity.class_index,
                "item_code": identity.item_code,
                "crops": counts[identity.legacy_class_name],
                "source_families": len({row["family_id"] for row in provenance}),
                "source_zips": len({row["source_zip"] for row in provenance}),
            }
        )
show_table(count_rows, "Phân bố AXE theo split và item code")
show_table(
    [
        {"artifact": path.relative_to(PROJECT_ROOT).as_posix(), "tồn tại": path.is_file(), "bytes": path.stat().st_size if path.is_file() else 0}
        for path in (crops_root / "crop_manifest.csv", clean_root / "class_mapping.json", clean_root / "source_family_manifest.json", clean_root / "build_report.json")
    ],
    "Artifact provenance",
)

In [ ]:
import matplotlib.pyplot as plt

item_codes = list(profile.item_codes)
train_counts = {row["item_code"]: row["crops"] for row in count_rows if row["split"] == "train"}
val_counts = {row["item_code"]: row["crops"] for row in count_rows if row["split"] == "val"}
positions = list(range(len(item_codes)))
width = 0.42
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar([x - width / 2 for x in positions], [train_counts[code] for code in item_codes], width, label="train")
ax.bar([x + width / 2 for x in positions], [val_counts[code] for code in item_codes], width, label="val")
ax.set_xticks(positions, item_codes, rotation=25, ha="right")
ax.set_ylabel("Số crop")
ax.set_title("Phân bố crop AXE 5 SKU")
ax.legend()
fig.tight_layout()
plt.show()

## Readiness check

Readiness xác minh file ảnh train/validation, mapping 5 SKU, exact-hash overlap, family overlap và fingerprint. Nó không nạp test query. Vì phải mở nhiều ảnh nên mặc định tắt. CLI huấn luyện cũng chạy lại guard này trước khi tạo model.

In [ ]:
if RUN_READINESS_CHECK:
    readiness = audit_training_readiness(crops_root, clean_root)
    show_table(
        [
            {
                "split": split,
                "crops": readiness["crop_counts"][split],
                "source_families": readiness["source_family_counts"][split],
            }
            for split in ("train", "val")
        ],
        "Kết quả readiness",
    )
    assert readiness["test_queries_loaded"] is False
else:
    readiness = None
    print("Readiness đang tắt: RUN_READINESS_CHECK=False")

## Huấn luyện thủ công

Ô này gọi CLI hiện có nên giữ nguyên seed, freeze/unfreeze, optimizer, scheduler, checkpoint policy và artifact schema. `TRAINING_RUN_NAME` phải mới; CLI không ghi đè và không tự resume.

In [ ]:
training_command = [
    sys.executable,
    str(resolve_repo_path("scripts/train_axe_5sku.py")),
    "--config", str(resolve_repo_path(TRAINING_CONFIG_PATH)),
    "--run-name", TRAINING_RUN_NAME,
]
if DEVICE_OVERRIDE is not None:
    training_command.extend(["--device", DEVICE_OVERRIDE])
print("Lệnh đã chuẩn bị:", subprocess.list2cmdline(training_command))
if RUN_TRAINING:
    if not RUN_READINESS_CHECK:
        raise RuntimeError("Bật RUN_READINESS_CHECK và xem kết quả trước khi huấn luyện.")
    subprocess.run(training_command, cwd=PROJECT_ROOT, check=True)
else:
    print("Không huấn luyện: RUN_TRAINING=False")

## Xem lại validation đã ghi

Training đã tính validation ở mỗi epoch và dùng đúng metric của workflow để chọn best checkpoint. Ô này chỉ đọc `metrics.csv`/`summary.json`; không chạy inference mới và không mở test.

In [ ]:
if RUN_VALIDATION_REVIEW:
    validation_run_dir = resolve_repo_path(VALIDATION_RUN_DIR)
    metrics_path = validation_run_dir / "metrics.csv"
    summary_path = validation_run_dir / "summary.json"
    with metrics_path.open("r", encoding="utf-8", newline="") as handle:
        history = list(csv.DictReader(handle))
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text(encoding="utf-8"))
        show_table([{"trường": key, "giá trị": value} for key, value in summary.items() if not isinstance(value, (dict, list))], "Tóm tắt validation")
    metric_name = "val_macro_f1" if training_mode == "classification_only" else "val_retrieval_r1_macro"
    epochs = [int(row["epoch"]) for row in history]
    values = [float(row[metric_name]) for row in history]
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(epochs, values, label=metric_name)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric_name)
    ax.set_title("Đường validation đã ghi trong run")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("Không đọc lại validation: RUN_VALIDATION_REVIEW=False")

## Test holdout — opt-in riêng

Chỉ sau khi checkpoint đã được chọn bằng validation mới đặt `RUN_TEST_EVALUATION=True`. Notebook tự chọn CLI classification hoặc retrieval theo `training.mode`; output phải chưa tồn tại. Test không được dùng để điều chỉnh hyperparameter hoặc chọn checkpoint.

In [ ]:
test_script = (
    "scripts/evaluate_axe_5sku_classification.py"
    if training_mode == "classification_only"
    else "scripts/evaluate_axe_5sku.py"
)
test_output = resolve_repo_path(TEST_OUTPUT_DIR)
test_command = [
    sys.executable,
    str(resolve_repo_path(test_script)),
    "--checkpoint", str(resolve_repo_path(TEST_CHECKPOINT)),
    "--dataset-dir", str(crops_root),
    "--output-dir", str(test_output),
]
if DEVICE_OVERRIDE is not None:
    test_command.extend(["--device", DEVICE_OVERRIDE])
print("Lệnh test đã chuẩn bị:", subprocess.list2cmdline(test_command))
if RUN_TEST_EVALUATION:
    if test_output.exists():
        raise FileExistsError(f"Chọn output mới; không ghi đè: {test_output}")
    subprocess.run(test_command, cwd=PROJECT_ROOT, check=True)
else:
    print("Không đánh giá test: RUN_TEST_EVALUATION=False")

## Sau khi chạy

Dùng `03_metrics_review.ipynb` để so sánh best/last và confusion matrix, rồi `04_gallery_and_inference.ipynb` để kiểm tra tương thích checkpoint–catalog–mapping–gallery. Huấn luyện embedding và thêm gallery item là hai quyết định riêng; một thương hiệu mới không bắt buộc retrain nếu embedding hiện tại đã được validation phù hợp, nhưng luôn cần dữ liệu gallery và kiểm tra chất lượng riêng.